---   
 <img align="left" width="75" height="75"  src="https://upload.wikimedia.org/wikipedia/en/c/c8/University_of_the_Punjab_logo.png"> 

<h1 align="center">Department of Data Science</h1>
<h1 align="center">Course: Tools and Techniques for Data Science</h1>

---
<h3><div align="right">Instructor: Muhammad Arif Butt, Ph.D.</div></h3>    

<h1 align="center">Lecture 6.15 (Hyperparameter Tuning using Grid and Random Search)</h1>

<a href="https://colab.research.google.com/github/arifpucit/data-science/blob/master/Section-4-Mathematics-for-Data-Science/Lec-4.1(Descriptive-Statistics).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <img align="center" width="900" src="images/ml-topimg1.png"  >

# Learning agenda of this notebook

### A Recap of What we have done so far?


<font color='green'> 
<br><br>
    
1. **Data Acquisition**
    - Libraries Built-in Datasets
        - Seaborn: (iris, titanic, tips, flights, panguins, car_crashes)
        - Scikit-learn: (iris, digits, diabetes, Bostan housing) 
        - NLTK: (movie-reviews, product_reviews, twitter_samples, gutenberg, genesis, timeit, voice, wordnet, sentiword)
    - Use Public Dataset Repositories:
        - https://www.kaggle.com/
        - https://data.gov/
        - https://archive.ics.uci.edu/ml/index.php
        - https://github.com/
    - Use Company's Database: (SQL, NOSQL, Data warehouse, Data lake)
    - Generate your own Datasets:
        - Use Web scraping or Web API
        - IoT Devices 
        - Crowd Sourcing (Amazon Mechanical Turk, Lionbridge AI)
        - Data Augmentation
<br><br>
2. **Data Preprocessing and Feature Engineering**
    - Detecting and handling outliers
    - Missing values Imputation
    - Encoding Categorical Features
    - Feature Scaling
    - Extracting Information 
    - Combining Information
<br><br>
3. **EDA and Visualization**
    - Matplotlib
    - Seaborn
    - Plotly
<br><br>
4. **Machine Learning Model Creation, Training and Evaluation**
    - Choosing the right estimator
    - Fit or train the model on training data
    - Do predictions on the test data
    - Evaluate the Model using simple train-test split and using Cross Validation Techniques
</font>

# Today's Agenda
- **Improve the Baseline Model**<br><br>
    - **From a `Data Perspective:`**
        - Could we collect more data samples?
        - Could we improve our data?
            - Do more cleaning and preprocessing
            - Add more relevent features (if possible)
            - Drop irrelevant features (if any)<br><br>
    - **From a `Model Perspective:`**
        - Choose some other estimator or model
        - Perform Hyperparameter Tuning of the current model
<br><br><br>

- **Model Parameters vs Hyperparameters**
- **Hyperparameters of Ridge Regression**
- **Ways to find the best hyperparameters for your model**
    - By hand manually using a loop
        - Using simple Train-Test split
        - Using KFold Cross Validation
    - Use GridSearchCV
    - Use RandomizedSearchCV

# 1. Overview of Hyperparameters

## a. Model Parameters vs Hyperparameters

|Ser | Hyperparameters | Model Parameters |
|:-:| :- | :- |
1 | Hyperparameters are set manually by ML engineer/practitioner prior to the start of the model’s training. | Model parameters are learnt by the learning algorithm during the training phase.|
2 | Hyperparameters are used to optimize machine learning model. |Model’s parameters are later used for prediction. |
3 | They are internal to the model. | Thy are external to the model. |Model’s parameters are later used for prediction. | 
4 | Examples: Value of K in KNN, learning rate for training a neural network, number of trees in RandomForrest. |Examples: Coefficients in a Linear or Logistic regression, support vectors in a support vector machine, and weights in an artificial neural network.| 

## b.  An Intuition of Hyperparameter Tuning
<img align="center" width="800" src="images/pt1.png"  >

## c. How to find Hyperparameters of a Model?
- Visit: https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html
- Research Problem: Understand, implement and compare different `solvers` and try designing a new solver.
- [A Survey of Optimization Methods From a Machine Learning Perspective](https://arxiv.org/pdf/1906.06821.pdf)

In [1]:
from sklearn.linear_model import Ridge
model = Ridge()
model.get_params()

{'alpha': 1.0,
 'copy_X': True,
 'fit_intercept': True,
 'max_iter': None,
 'normalize': 'deprecated',
 'positive': False,
 'random_state': None,
 'solver': 'auto',
 'tol': 0.001}

<img align="center" width="800" src="images/regularization-main.png"  >

## d. How to find best hyperparameters for our model

**Hyperparameter Tuning Techniques:**
- `By Hand:` Select the hyperparameters values based on intuition/experience/guessing, train the model with the hyperparameters, and score on the validation data. Repeat process until you run out of patience or are satisfied with the results. 
- `GridSearchCV:` Set up a grid of hyperparameter values and for each combination, train a model and score on the validation data. In this approach, every single combination of hyperparameters values is tried which can be very inefficient!
- `RandomizedSearchCV:` Set up a grid of hyperparameter values and select random combinations to train the model and score. The number of search iterations is set based on time/resources.


**Hypertuning Steps**
- Make a list of different hyperparameters based on the problem in hand. If there are more than one hyperparameter then make grid with different combination of parameters
- Fit all of them separately to the model. 
- Note down the model performance
- Choose the best performing one
- Always use cross validation technique for hyperparameter tuning to avoid the model overfitting on test data.


# 3. Find Optimized Hyperparameter By Hand

## a. Find Optimized Hyperparameter by Trial and Error
- Approach 1: Use train_test_split and manually tune parameters by trial and error

**Load the Advertising Dataset:**

In [2]:
import pandas as pd
import numpy as np
df = pd.read_csv("datasets/advertising4D.csv")
df

,TV,radio,newspaper,sales
0,230.1,37.8,69.2,22.1
1,44.5,39.3,45.1,10.4
2,17.2,45.9,69.3,9.3
3,151.5,41.3,58.5,18.5
4,180.8,10.8,58.4,12.9
...,...,...,...,...
195,38.2,3.7,13.8,7.6
196,94.2,4.9,8.1,9.7
197,177.0,9.3,6.4,12.8
198,283.6,42.0,66.2,25.5


**Do a Train-Test Split:**

In [3]:
from sklearn.model_selection import train_test_split

X = df.drop('sales', axis=1)
y = df['sales']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True)

**Scale the Data:**

In [4]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_train)

X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

**Instentiate the model `Ridge()`, Train and Evaluate**

In [6]:
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

# Instentiate a model and fit it to training data with default hyperparameters
model = Ridge()
model.fit(X_train, y_train)

# Evaluate
r2 = r2_score(y_test, model.predict(X_test))
r2 = model.score(X_test, y_test)
print("R2 Score: ", r2)

R2 Score:  0.8894639826365438


**Adjust Parameters `Ridge(alpha=1.0, solver='auto')`, Train and Re-evaluate**

In [7]:
# Adjust values of hyperparameters
model = Ridge(alpha=1.0, solver='auto')
# Retrain
model.fit(X_train, y_train)
# Re-evaluate
r2 = r2_score(y_test, model.predict(X_test))
r2 = model.score(X_test, y_test)
print("R2 Score: ", r2)

R2 Score:  0.8894639826365438


**Adjust Parameters `Ridge(alpha=1000, solver='auto')`, Train and Re-evaluate**

In [8]:
# Adjust values of hyperparameters
model = Ridge(alpha=1000, solver='auto')
# Retrain
model.fit(X_train, y_train)
# Re-evaluate
r2 = r2_score(y_test, model.predict(X_test))
r2 = model.score(X_test, y_test)
print("R2 Score: ", r2)

R2 Score:  0.24159725555280676


**Adjust Parameters `Ridge(alpha=100, solver='lsqr')`, Train and Re-evaluate**

In [9]:
# Adjust values of hyperparameters
model = Ridge(alpha=100, solver='lsqr')
# Retrain
model.fit(X_train, y_train)

# Re-evaluate
r2 = r2_score(y_test, model.predict(X_test))
r2 = model.score(X_test, y_test)
print("R2 Score: ", r2)

R2 Score:  0.7554991523571211


>- Above approach is tiresome and very manual.
>- Moreover, you get a different score each time you train the model on a different split

In [21]:
df = pd.read_csv("datasets/advertising4D.csv")
X = df.drop('sales', axis=1)
y = df['sales']

# Do a train-test-split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True)

# SCALE DATA
scaler = StandardScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

# Specify values of hyperparameters
model = Ridge(alpha=100, solver='lsqr')

# Train
model.fit(X_train, y_train)

# Evaluate
r2 = r2_score(y_test, model.predict(X_test))
r2 = model.score(X_test, y_test)
print("R2 Score: ", r2)

R2 Score:  0.7665207268336094


>- Each time you execute the above code cell, you get different R2 scores. None of these R2 scores is the true representation of the entire dataset. This is because of the train-test split. The limitation is each time the model is tested on different 20% of the test set. The solution is to use some Cross-Validation technique.

### Use of `cross_val_score()` Method (KFold)
<img align="center" width="800" src="images/gs2a.png"  >

In [22]:
from sklearn.model_selection import cross_val_score
df = pd.read_csv("datasets/advertising4D.csv")
X = df.drop('sales', axis=1)
y = df['sales']
# Do a train-test-split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True)
# SCALE DATA
scaler = StandardScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)


#cross_val_score() is a cross validation method that trains and tests a model over multiple folds of a dataset. 
cv_scores = cross_val_score(
                    estimator = Ridge(alpha=100, solver='lsqr'), 
                    X = X_train, 
                    y = y_train, 
                    scoring = 'r2', # Can specify scoring metric of your choice for the estimator
                    cv = 5  #An integer value means KFold Cross Validation (you can pass CV object of your choice)
                          )
print("R2 scores for all the folds: ", cv_scores)
print("Mean R2 score: ", np.mean(cv_scores))

R2 scores for all the folds:  [0.820465   0.7314995  0.66893367 0.70452143 0.73488347]
Mean R2 score:  0.7320606116327646


## b. Find Optimized Hyperparameter using For Loops

### (i) Use Simple Train-Test-Split

In [23]:
df = pd.read_csv("datasets/advertising4D.csv")
X = df.drop('sales', axis=1)
y = df['sales']
# Do a train-test-split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True)
# SCALE DATA
scaler = StandardScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)


alpha_list = [1000, 100, 10, 5, 1, 0.8, 0.5, 0.2]
solver_list = ['lsqr', 'svd']

# Find the best parameters out of the above two lists w/o using cross validation
for a in alpha_list: 
    for s in solver_list: 
        model = Ridge(alpha=a, solver=s)# Instentiate model each time with different values of hyperparameters
        model.fit(X_train, y_train)     # Fit the model to training data
        r2 = model.score(X_test, y_test)  # Evaluate by calculating R2 Score
        print(a, s,":",r2)

1000 lsqr : 0.2183650889581339
1000 svd : 0.21837890617440447
100 lsqr : 0.7703406718546536
100 svd : 0.7703406718546536
10 lsqr : 0.9195082041247284
10 svd : 0.9195082041247286
5 lsqr : 0.9234782009903446
5 svd : 0.9234782009903447
1 lsqr : 0.9255955273333893
1 svd : 0.9255955273333893
0.8 lsqr : 0.9256731113053106
0.8 svd : 0.9256731113053106
0.5 lsqr : 0.9257840887825026
0.5 svd : 0.9257840887825026
0.2 lsqr : 0.9258885096032944
0.2 svd : 0.9258885096032944


> **Limitation:**
>- Model evaluation is done on just 20% of the data.
>- After finalizing the best combination of hyperparameters, we are not left with any unseen data on which we can do the final evaluation of the model with the best hyperparameters

### (ii) Use Cross Validation

In [24]:
df = pd.read_csv("datasets/advertising4D.csv")
X = df.drop('sales', axis=1)
y = df['sales']
# Do a train-test-split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True)
# SCALE DATA
scaler = StandardScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)


alpha_list = [1000, 100, 10, 5, 1, 0.8, 0.5, 0.2]
solver_list = ['lsqr', 'svd']

# Find the best parameters out of the above two lists using cross validation
for a in alpha_list: 
    for s in solver_list: 
        cv_scores = cross_val_score(Ridge(alpha=a, solver=s ), X_train, y_train, scoring='r2', cv=5)
        print(a, s,":",np.mean(cv_scores))
        
# Once the model is finalized and its hyperparameters tuned, 
# we use the X_test and y_test dataset (still unused/unseen to model) for evaluation of the finalized model

1000 lsqr : 0.16663364953249218
1000 svd : 0.1666395195396047
100 lsqr : 0.7047157350669636
100 svd : 0.7047157350669636
10 lsqr : 0.8755540085968858
10 svd : 0.8755540085968858
5 lsqr : 0.8793597246934309
5 svd : 0.8793597246934309
1 lsqr : 0.8807935636226911
1 svd : 0.880793563622691
0.8 lsqr : 0.8808212506440828
0.8 svd : 0.8808212506440828
0.5 lsqr : 0.8808542864165261
0.5 svd : 0.8808542864165261
0.2 lsqr : 0.8808769833864989
0.2 svd : 0.8808769833864989


# 4. Find Optimized Hyperparameters using `GridSearchCV()`

- GridSearchCV Documentation: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html

<img align="right" width="400" src="images/gridsearch.png"  >

- Grid search is a method for hyperparameter optimization that involves specifying a list of values for each hyperparameter that you want to optimize, and then training a model for each combination of these values.
- Basically, we divide the domain of the hyperparameters into a discrete grid. 
- Then, we try every combination of values of this grid, calculating some performance metrics using cross-validation. 
- The point of the grid that maximizes the average value in cross-validation, is the optimal combination of values for the hyperparameters.
- Additionally, it is recommended to use cross-validation when performing hyperparameter optimization. This can provide a more accurate estimate of the model’s performance and help to avoid overfitting.

In [25]:
from sklearn.model_selection import  GridSearchCV

df = pd.read_csv("datasets/advertising4D.csv")
X = df.drop('sales', axis=1)
y = df['sales']
# Do a train-test-split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True)
# SCALE DATA
scaler = StandardScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

#Dictionary with parameters names (`str`) as keys and lists of parameter settings to try as values
params = { 'alpha': [1000, 100, 10, 1, 0.5],
           'solver': ['lsqr', 'svd'] }

# Find the best parameters out of the above dictionary using GridSearchCV object
gs = GridSearchCV(estimator=Ridge(), 
                   param_grid=params,
                   scoring='r2',
                   cv=5,
                   n_jobs=-1) 

gs.fit(X_train, y_train)

#Attributes of gs object
print("Best Score: ", gs.best_score_)
print("Best Score: ", gs.best_params_)

Best Score:  0.8712355992402852
Best Score:  {'alpha': 1, 'solver': 'lsqr'}


In [ ]:
#Check out the attributes and methods of this trained gs object
print(dir(gs))

In [ ]:
# cv_results_ attribute of gs is a dictionary object containing following information
gs.cv_results_

In [ ]:
# For better readability let us display the dictionary object as a dataframe
df = pd.DataFrame(gs.cv_results_)
df

In [ ]:
df.loc[:, ['param_alpha', 'param_solver', 'mean_test_score']]

**Limitations of GridSearchCV:**
- Grid search is an exhaustive algorithm that spans all the combinations, so it can actually find the best point in the domain. 
- For that it trains a separate model for every combination of hyperparameter values.
- Suppose you have million of data points in your dataset and a bundle of hyperparameters and their values to tune. In that case the grid will be multidimensional and the algorithm will become computationally expensive as well as time consuming.

# 5. Find Optimized Hyperparameters using `RandomizedSearchCV()`


- RandomizedSearchCV Documentation: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html

<img align="right" width="400" src="images/randomsearch.png"  >

- Random search is similar to grid search, but instead of using all the points in the grid, it tests only a randomly selected subset of these points. <br><br>
- The smaller this subset, the faster but less accurate the optimization. The larger this dataset, the more accurate the optimization but the closer to a grid search.<br><br>
- Random search is a very useful option when you have several hyperparameters with a fine-grained grid of values. 

In [26]:
from sklearn.model_selection import RandomizedSearchCV

df = pd.read_csv("datasets/advertising4D.csv")
X = df.drop('sales', axis=1)
y = df['sales']
# Do a train-test-split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True)
# SCALE DATA
scaler = StandardScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)  

#Dictionary with parameters names (`str`) as keys and lists of parameter settings to try as values
params = { 'alpha': [1000, 100, 10, 1, 0.5],
           'solver': ['lsqr', 'svd'] }

# Find the best parameters out of the above dictionary using RandomizedSearchCV object
rs = RandomizedSearchCV(estimator=Ridge(), 
                   param_distributions=params,
                   n_iter=6, #Number of parameter combinations to try.
                   scoring='r2',
                   cv=5,
                   n_jobs=-1)  

rs.fit(X_train, y_train)

print("Best Score: ", rs.best_score_)
print("Best Score: ", rs.best_params_)

Best Score:  0.8753981605897158
Best Score:  {'solver': 'lsqr', 'alpha': 1}


In [27]:
#Check out the attributes and methods of this trained rs object
print(dir(rs))

['__abstractmethods__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_abc_impl', '_check_feature_names', '_check_n_features', '_check_refit_for_multimetric', '_estimator_type', '_format_results', '_get_param_names', '_get_tags', '_more_tags', '_pairwise', '_repr_html_', '_repr_html_inner', '_repr_mimebundle_', '_required_parameters', '_run_search', '_select_best_index', '_validate_data', 'best_estimator_', 'best_index_', 'best_params_', 'best_score_', 'classes_', 'cv', 'cv_results_', 'decision_function', 'error_score', 'estimator', 'fit', 'get_params', 'inverse_transform', 'multimetric_', 'n_features_in_', 'n_iter', 'n_jobs', 'n_splits_', 'param_distributions'

In [28]:
# cv_results_ attribute of rs is a dictionary object containing following information
rs.cv_results_

{'mean_fit_time': array([0.00078545, 0.00050902, 0.0003314 , 0.00059481, 0.00052729,
        0.00058851]),
 'std_fit_time': array([9.72188527e-05, 1.12828982e-04, 6.40632097e-06, 2.71033351e-04,
        1.64775880e-04, 1.06290479e-04]),
 'mean_score_time': array([0.00039911, 0.00024357, 0.00017958, 0.00022726, 0.00021191,
        0.00025144]),
 'std_score_time': array([1.05277287e-04, 6.94193023e-05, 3.20014224e-06, 8.28692223e-05,
        5.11271730e-05, 5.16066033e-05]),
 'param_solver': masked_array(data=['lsqr', 'svd', 'svd', 'lsqr', 'lsqr', 'lsqr'],
              mask=[False, False, False, False, False, False],
        fill_value='?',
             dtype=object),
 'param_alpha': masked_array(data=[1000, 10, 100, 100, 1, 10],
              mask=[False, False, False, False, False, False],
        fill_value='?',
             dtype=object),
 'params': [{'solver': 'lsqr', 'alpha': 1000},
  {'solver': 'svd', 'alpha': 10},
  {'solver': 'svd', 'alpha': 100},
  {'solver': 'lsqr', 'alpha': 

In [29]:
# For better readability let us display the dictionary object as a dataframe
df = pd.DataFrame(rs.cv_results_)
df

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_solver,param_alpha,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.000785,0.000097,0.000399,0.000105,lsqr,1000,"{'solver': 'lsqr', 'alpha': 1000}",0.102338,0.227708,0.205684,0.216100,0.086417,0.167649,0.060441,6
1,0.000509,0.000113,0.000244,0.000069,svd,10,"{'solver': 'svd', 'alpha': 10}",0.763402,0.864178,0.923191,0.907879,0.897011,0.871132,0.057244,3
2,0.000331,0.000006,0.000180,0.000003,svd,100,"{'solver': 'svd', 'alpha': 100}",0.593993,0.755794,0.760173,0.760567,0.687921,0.711689,0.064965,4
3,0.000595,0.000271,0.000227,0.000083,lsqr,100,"{'solver': 'lsqr', 'alpha': 100}",0.593993,0.755794,0.760173,0.760567,0.687921,0.711689,0.064965,4
4,0.000527,0.000165,0.000212,0.000051,lsqr,1,"{'solver': 'lsqr', 'alpha': 1}",0.773437,0.862089,0.923470,0.909049,0.908946,0.875398,0.055031,1
5,0.000589,0.000106,0.000251,0.000052,lsqr,10,"{'solver': 'lsqr', 'alpha': 10}",0.763402,0.864178,0.923191,0.907879,0.897011,0.871132,0.057244,2
